#**STEP 1 :- IMPORT LIBRARIES**

In [ ]:
# ==========================================
# IMPORTING LIBRARIES
# ==========================================

import numpy as np
import pandas as pd
import json
import random
import pickle
import re

# Deep Learning Libraries
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.models import load_model

from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Bidirectional

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.utils import to_categorical

# Machine Learning Utilities
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Visualization Libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

print("All Libraries Loaded Successfully")

All Libraries Loaded Successfully


# **STEP 2 :- LOAD DATASET**

In [ ]:
# ==========================================
# LOADING INTENTS DATASET
# ==========================================

with open("intents_bot.json", "r") as file:
    data = json.load(file)

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


# **STEP 3 :- DATA PREPARATION**

In [ ]:
# ==========================================
# DATA PREPROCESSING
# ==========================================

sentences = []
labels = []

for intent in data["intents"]:

    tag = intent["tag"]

    for pattern in intent["patterns"]:

        pattern = pattern.lower()

        sentences.append(pattern)
        labels.append(tag)

print("Total Sentences :", len(sentences))
print("Total Labels :", len(labels))

Total Sentences : 113
Total Labels : 113


# **STEP 4 :- TOKENIZATION**

In [ ]:
# ==========================================
# TOKENIZATION
# ==========================================

vocab_size = 5000
max_len = 25

tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(sentences)

sequences = tokenizer.texts_to_sequences(
    sentences
)

X = pad_sequences(
    sequences,
    maxlen=max_len,
    padding="post"
)

print("Input Shape :", X.shape)

Input Shape : (113, 25)


# **STEP 5 :- LABEL ENCODING**

In [ ]:
# ==========================================
# LABEL ENCODING
# ==========================================

encoder = LabelEncoder()

y = encoder.fit_transform(labels)

y = to_categorical(y)

print("Output Shape :", y.shape)

Output Shape : (113, 30)


# STEP 6 :- TRAIN-TEST SPLIT

In [ ]:
# ==========================================
# TRAIN TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Samples :", len(X_train))
print("Testing Samples :", len(X_test))

Training Samples : 90
Testing Samples : 23


# **STEP 7 :- DEEP LEARNING LSTM MODEL**

In [ ]:
# ==========================================
# BUILDING LSTM MODEL
# ==========================================

model = Sequential()

# Embedding Layer
model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_length=max_len
    )
)

# First Bi-LSTM Layer
model.add(
    Bidirectional(
        LSTM(
            128,
            return_sequences=True
        )
    )
)

model.add(
    Dropout(0.3)
)

# Second LSTM Layer
model.add(
    LSTM(
        64,
        return_sequences=False
    )
)

model.add(
    Dropout(0.3)
)

# Dense Layers
model.add(
    Dense(
        64,
        activation="relu"
    )
)

model.add(
    Dense(
        32,
        activation="relu"
    )
)

# Output Layer
model.add(
    Dense(
        y.shape[1],
        activation="softmax"
    )
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# **STEP 8 :- COMPILE MODEL**

In [ ]:
# ==========================================
# MODEL COMPILATION
# ==========================================

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# **STEP 9 :- TRAIN MODEL**

In [ ]:
# ==========================================
# TRAINING MODEL
# ==========================================

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=16,
    verbose=1
)

Epoch 1/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 7s 204ms/step - accuracy: 0.0333 - loss: 3.4057 - val_accuracy: 0.0000e+00 - val_loss: 3.4143
Epoch 2/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.0556 - loss: 3.4011 - val_accuracy: 0.0435 - val_loss: 3.4261
Epoch 3/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.0444 - loss: 3.3872 - val_accuracy: 0.0435 - val_loss: 3.4361
Epoch 4/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 76ms/step - accuracy: 0.0667 - loss: 3.3865 - val_accuracy: 0.0435 - val_loss: 3.4534
Epoch 5/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 95ms/step - accuracy: 0.0444 - loss: 3.3688 - val_accuracy: 0.0435 - val_loss: 3.4947
Epoch 6/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 124ms/step - accuracy: 0.0556 - loss: 3.3662 - val_accuracy: 0.0435 - val_loss: 3.5409
Epoch 7/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 138ms/step - accuracy: 0.0000e+00 - loss: 3.3544 - val_accuracy: 0.0435 - val_loss: 3.5296
Epoch 8/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 142ms/step - accuracy: 0.0556 - loss: 3.3468 - val_accuracy: 0.0435

# **STEP 10 :- SAVE MODEL**

In [ ]:
# ==========================================
# SAVE MODEL
# ==========================================

model.save("MLBOT_LSTM_Model.keras")

pickle.dump(
    tokenizer,
    open("tokenizer.pkl", "wb")
)

pickle.dump(
    encoder,
    open("label_encoder.pkl", "wb")
)

print("Model Saved Successfully")

Model Saved Successfully


 # **STEP 11: PREDICTION FUNCTION**

In [ ]:
# ==========================================
# CHATBOT RESPONSE FUNCTION
# ==========================================

def predict_class(text):

    text = text.lower()

    sequence = tokenizer.texts_to_sequences([text])

    padded = pad_sequences(
        sequence,
        maxlen=max_len,
        padding="post"
    )

    prediction = model.predict(
        padded,
        verbose=0
    )

    index = np.argmax(prediction)

    tag = encoder.inverse_transform([index])[0]

    return tag

# **STEP 12 :- CHAT LOOP**

In [ ]:
# ==========================================
# CHAT LOOP
# ==========================================

print("MLBOT Started")
print("Type 'quit' to exit")

while True:

    user_input = input("You : ")

    if user_input.lower() == "quit":
        break

    tag = predict_class(user_input)

    response = "Sorry, I don't understand."

    for intent in data["intents"]:

        if intent["tag"] == tag:
            response = random.choice(
                intent["responses"]
            )

    print("Bot :", response)

MLBOT Started
Type 'quit' to exit
You : Hi
Bot : Date and Time
You : hi
Bot : Date and Time
You : quit
